# Let's test our kafka listener


# Swift

<img src="images/swift.png" alt="Swift and Fermi mission comparison" width="600"/>


<img src="images/swift_1.png" alt="Swift and Fermi mission comparison" width="600"/>


<img src="images/xrt.png" alt="Swift and Fermi mission comparison" width="600"/>


<img src="images/uvot.png" alt="Swift and Fermi mission comparison" width="600"/>


Check https://gcn.gsfc.nasa.gov/swift.html for further details for the content of each notice

# Fermi


<img src="images/fermi.png" alt="Swift and Fermi mission comparison" width="600"/> 




<img src="images/fermi_1.png" alt="Swift and Fermi mission comparison" width="600"/>

<img src="images/fermi_2.png" alt="Swift and Fermi mission comparison" width="600"/>

Check https://gcn.gsfc.nasa.gov/fermi.html for further details for the content of each notice

## Get familiar with notice content

https://gcn.gsfc.nasa.gov/fermi_grbs.html

https://gcn.gsfc.nasa.gov/swift_grbs.html


## Let's try our first kafka connection

set 

```python
"auto.offset.reset": "earliest"
```

to receive all the messages you missed since the last time you had the listener active

set 

```python
"auto.offset.reset": "latest" 
```

to receive ONLY NEW  messages you missed since the last time you had the listener active


In [1]:
from gcn_kafka import Consumer
from confluent_kafka import TopicPartition
import os
import json
import datetime 
import xmltodict
from astropy.coordinates import SkyCoord

# Connect as a consumer (client "Mac MOC")
# Warning: don't share the client secret with others.
CONFIG = {"group.id": "", "auto.offset.reset": "earliest"}
consumer = Consumer(config=CONFIG,
                    client_id='5h42prl0v1gtqh60b2p2a1ltho',
                    client_secret='icq2qm2nelng5ucelm71dh9kf3l0jgq0b0omthv457i5h74im0v',
                    domain='gcn.nasa.gov')


In [30]:
import email

def write_json(text_data, output_dir="./gcn_alerts"):
    """Parse GCN text format and extract relevant information, then save to JSON."""
    
    # Convert bytes to string if necessary
    if isinstance(text_data, bytes):
        text_data = text_data.decode('utf-8')
    
    # Parse using email module
    msg = email.message_from_string(text_data)
    result = dict(msg)
    
    try:
        # Create output directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)
        
        # Create filename using trigger number or timestamp
        if 'TRIGGER_NUM' in result:
            filename = f"GRB_{result['TRIGGER_NUM']}.json"
        else:
            timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"GRB_{timestamp}.json"
        
        filepath = os.path.join(output_dir, filename)
        
        with open(filepath, 'w') as f:
            json.dump(result, f, indent=2)
        
        print(f"Saved to: {filepath}")
    
    except Exception as e:
        print("Error processing GCN text notice:", e)
    
    return result


In [19]:
# Time ranges are specified as Kafka times (milliseconds since the unix epoch)

t1 = 1

t2 = 1

timestamp1 = int((datetime.datetime.now() - datetime.timedelta(days=t1)).timestamp() * 1000)
timestamp2 = timestamp1 + t2*86400000 # +7 days

In [ ]:


print(f"Fetching messages between {timestamp1} and {timestamp2}")

# Subscribe to topics and receive alerts
topics = 'gcn.classic.text.FERMI_GBM_ALERT'
# topics = 'gcn.classic.text.SWIFT_BAT_GRB_POS_ACK'
# topics = 'gcn.classic.text.SWIFT_XRT_POSITION'
# topics = 'gcn.classic.text.SWIFT_UVOT_POS'

start = consumer.offsets_for_times(
    [TopicPartition(topics, 0, timestamp1)])
end = consumer.offsets_for_times(
    [TopicPartition(topics, 0, timestamp2)])


consumer.assign(start)

# Calculate the number of messages to consume, ensuring it's within valid range
num_messages = end[0].offset - start[0].offset


for message in consumer.consume(abs(num_messages), timeout=1):
    
# while True:
#     for message in consumer.consume(timeout=1):
        if message.error():
            print(message.error())
            continue
        # Print the topic and message ID
        print(f'topic={message.topic()}, offset={message.offset()}')
        value = message.value()
        # xml_dict = xmltodict.parse(value)
        write_json(value)

Fetching messages between 1761573052660 and 1762437052660
topic=gcn.classic.text.FERMI_GBM_ALERT, offset=3458
Saved to: ./gcn_alerts/GRB_783741831.json
topic=gcn.classic.text.FERMI_GBM_ALERT, offset=3459
Saved to: ./gcn_alerts/GRB_783773800.json
topic=gcn.classic.text.FERMI_GBM_ALERT, offset=3460
Saved to: ./gcn_alerts/GRB_783810958.json
topic=gcn.classic.text.FERMI_GBM_ALERT, offset=3461
Saved to: ./gcn_alerts/GRB_783820740.json
topic=gcn.classic.text.FERMI_GBM_ALERT, offset=3462
Saved to: ./gcn_alerts/GRB_783822281.json
topic=gcn.classic.text.FERMI_GBM_ALERT, offset=3463
Saved to: ./gcn_alerts/GRB_783837993.json
topic=gcn.classic.text.FERMI_GBM_ALERT, offset=3464
Saved to: ./gcn_alerts/GRB_783844165.json
topic=gcn.classic.text.FERMI_GBM_ALERT, offset=3465
Saved to: ./gcn_alerts/GRB_783881201.json
topic=gcn.classic.text.FERMI_GBM_ALERT, offset=3466
Saved to: ./gcn_alerts/GRB_783882332.json
topic=gcn.classic.text.FERMI_GBM_ALERT, offset=3467
Saved to: ./gcn_alerts/GRB_783911182.json
to

# SVOM

<img src="images/svom.png" alt="Swift and Fermi mission comparison" width="600"/>

<img src="images/svom_1.png" alt="Swift and Fermi mission comparison" width="600"/>

<img src="images/svom_2.png" alt="Swift and Fermi mission comparison" width="600"/>

In [6]:

def voevent_to_json(xml_file):

    # Convert to normal dict and JSON
    json_data = json.loads(json.dumps(xml_file))

    # Navigate through the XML structure safely
    voevent = json_data.get("voe:VOEvent", {})
    wherewhen = voevent.get("WhereWhen", {})

    # Initialize result dictionary
    result = {}

    try:
        obs_location = wherewhen.get("ObsDataLocation", {}).get("ObservationLocation", {})
        astro_coords = obs_location.get("AstroCoords", {})
        pos = astro_coords.get("Position2D", {}).get("Value2", {})
        c1 = pos.get("C1")
        c2 = pos.get("C2")
        
        # The coordinates are already in RA/Dec format (not galactic)
        ra = float(c1)
        dec = float(c2)
        
        result['ra'] = ra
        result['dec'] = dec
        
        print("RA:", ra)
        print("Dec:", dec)

    except Exception as e:
        print("No coordinates found")

    trigger_time = astro_coords.get("Time", {}).get("TimeInstant", {}).get("ISOTime")
    result['trigger_time'] = trigger_time

    print("Trigger time:", trigger_time)

    # Save to JSON file
    output_dir = "./gcn_alerts"
    os.makedirs(output_dir, exist_ok=True)

    # Extract burst ID for filename
    burst_id = voevent.get("What", {}).get("Group", [{}])[0].get("Param", [{}])[1].get("@value", "unknown")
    filename = f"SVOM_{burst_id}.json"
    filepath = os.path.join(output_dir, filename)

    with open(filepath, 'w') as f:
        json.dump(result, f, indent=2)

    print(f"Saved to: {filepath}")

    return json_data

In [7]:
# Subscribe to topics and receive alerts
# topics = 'gcn.notices.svom.voevent.eclairs'
topics = 'gcn.notices.svom.voevent.grm'

start = consumer.offsets_for_times(
    [TopicPartition(topics, 0, timestamp1)])
end = consumer.offsets_for_times(
    [TopicPartition(topics, 0, timestamp2)])


consumer.assign(start)
for message in consumer.consume(end[0].offset - start[0].offset, timeout=1):
    
# while True:
#     for message in consumer.consume(timeout=1):
        if message.error():
            print(message.error())
            continue
        # Print the topic and message ID
        print(f'topic={message.topic()}, offset={message.offset()}')
        value = message.value()
        xml_dict = xmltodict.parse(value)
        voevent_to_json(xml_dict)

topic=gcn.notices.svom.voevent.grm, offset=199
No coordinates found
Trigger time: 2025-10-28T03:26:33
Saved to: ./gcn_alerts/SVOM_sb25102801.json
topic=gcn.notices.svom.voevent.grm, offset=200
No coordinates found
Trigger time: 2025-11-03T04:46:27
Saved to: ./gcn_alerts/SVOM_sb25110302.json
topic=gcn.notices.svom.voevent.grm, offset=201
RA: 136.6644
Dec: 14.9273
Trigger time: 2025-11-03T04:54:27.900000
Saved to: ./gcn_alerts/SVOM_sb25110303.json
topic=gcn.notices.svom.voevent.grm, offset=202
No coordinates found
Trigger time: 2025-11-03T09:54:44
Saved to: ./gcn_alerts/SVOM_sb25110304.json
topic=gcn.notices.svom.voevent.grm, offset=203
No coordinates found
Trigger time: 2025-11-03T12:59:30
Saved to: ./gcn_alerts/SVOM_sb25110306.json
topic=gcn.notices.svom.voevent.grm, offset=204
No coordinates found
Trigger time: 2025-11-03T16:21:04
Saved to: ./gcn_alerts/SVOM_sb25110308.json
topic=gcn.notices.svom.voevent.grm, offset=205
No coordinates found
Trigger time: 2025-11-05T00:58:34.600000
Sav

check https://fsc.svom.org/readthedocs/svom/notices_and_circulars/index.html for more details about circulars and notices

# Einstein Probe

<img src="images/ep.png" width="600"/>

<img src="images/ep_1.png"  width="600"/>

In [8]:
def to_json(text_data, output_dir="./gcn_alerts"):
    """Parse GCN JSON format and extract position and trigger time, then save to JSON."""
    
    try:
        # Parse the JSON data
        data = json.loads(text_data)
        
        # Extract only position and trigger time
        alert_info = {
            "trigger_time": data.get("trigger_time"),
            "ra": data.get("ra"),
            "dec": data.get("dec")
        }
        
        # Create output directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)
        
        # Generate filename based on trigger time and ID
        trigger_id = data.get("id", ["unknown"])[0]
        filename = f"einstein_probe_{trigger_id}.json"
        filepath = os.path.join(output_dir, filename)
        
        # Save to JSON file
        with open(filepath, 'w') as f:
            json.dump(alert_info, f, indent=2)
        
        print(f"Saved alert to {filepath}")
        return alert_info
        
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
        return None
    except Exception as e:
        print(f"Error processing alert: {e}")
        return None



In [9]:

# Subscribe to topics and receive alerts
topics = 'gcn.notices.einstein_probe.wxt.alert'


start = consumer.offsets_for_times(
    [TopicPartition(topics, 0, timestamp1)])
end = consumer.offsets_for_times(
    [TopicPartition(topics, 0, timestamp2)])

consumer.assign(start)

# Calculate the number of messages to consume, ensuring it's within valid range
num_messages = end[0].offset - start[0].offset

print(end[0].offset, start[0].offset)

for message in consumer.consume(abs(num_messages), timeout=1):
    
# while True:
#     for message in consumer.consume(timeout=1):
        if message.error():
            print(message.error())
            continue
        # Print the topic and message ID
        print(f'topic={message.topic()}, offset={message.offset()}')
        value = message.value()
        # xml_dict = xmltodict.parse(value)
        to_json(value)

181 179
topic=gcn.notices.einstein_probe.wxt.alert, offset=179
Saved alert to ./gcn_alerts/einstein_probe_01709247588.json
topic=gcn.notices.einstein_probe.wxt.alert, offset=180
Saved alert to ./gcn_alerts/einstein_probe_11900459650.json


# Let's switch on our kafka listener

In [ ]:
import threading
import time

consumer.subscribe(['gcn.notices.einstein_probe.wxt.alert',
                    'gcn.notices.svom.voevent.grm',
                    'gcn.notices.svom.voevent.eclairs',
                    'gcn.classic.text.SWIFT_BAT_GRB_POS_ACK',
                    'gcn.classic.text.FERMI_GBM_ALERT'])

def kafka_listener():

    try:
        while not stop_event.is_set():
            msg = consumer.poll(timeout=1.0)  # poll frequently
            if msg is None:
                continue
            if msg.error():
                print(f"Consumer error: {msg.error()}")
                continue

            for message in consumer.consume(timeout=1):
                if message.error():
                    print(message.error())
                    continue
                print(f'topic={message.topic()}, offset={message.offset()}')
                if message.topic() == 'gcn.notices.einstein_probe.wxt.alert':
                    value = message.value()
                    to_json(value)
                elif 'svom' in message.topic():
                    value = message.value()
                    xml_dict = xmltodict.parse(value)
                    voevent_to_json(xml_dict)
                elif 'SWIFT' in message.topic() or 'FERMI' in message.topic():
                    value = message.value()
                    write_json(value)

    except KafkaException as e:
        print(f"Kafka exception: {e}")
    finally:
        consumer.close()
        print("Consumer closed.")

## Start the listener as a background process

In [37]:
import threading
import time

# --- Stop event to safely stop the listener ---
stop_event = threading.Event()

# --- Start listener in a daemon thread ---
listener_thread = threading.Thread(target=kafka_listener, daemon=True)
listener_thread.start()

## Check if the listener is alive

In [ ]:
listener_thread.is_alive()

In [35]:

stop_event.set()  # This will stop keep_polling safely
listener_thread.join()    # wait for it to exit

# IGWN alerts


In [2]:

def parse_gw_notice(record):
    record = json.loads(record)

    # print(datetime.datetime.now(), record['superevent_id'], 'received')

    try:

        if record['event']['significant']==True and record['superevent_id'][0] != 'M':
            # this is a significant event
            print('Significant event detected:', record['superevent_id'])

        if record['superevent_id'][0] == 'M':
            # print('Mock event detected:', record['superevent_id'])

            return
        
        if record['alert_type'] == 'RETRACTION':
            print(record['superevent_id'], 'was retracted')
            return

        if record['alert_type'] == 'EARLYWARNING':
            print(datetime.datetime.now(), record['superevent_id'], 'EARLYWARNING')

        if record.get('external_coinc') is not None:
            print(datetime.datetime.now(), record['superevent_id'], 'RAVEN_ALERT')

        if record['event']['search'] == 'SSM':
            print(datetime.datetime.now(), record['superevent_id'], 'SSM event')
            HasNS = record['event']['properties']['HasNS']
            HasSSM = record['event']['properties']['HasSSM']
            HasMassGap = record['event']['properties']['HasMassGap']
            print('HasNS:', HasNS, 'HasSSM:', HasSSM, 'HasMassGap:', HasMassGap)

        #this far is not corrected by trials factor
        #meaning that we receive on average more alerts than 1/month
        if record['event']['group'] == 'Burst':
            far_min=1/(7*86400)
        else:
            far_min=1/(30*86400)

        far=record['event']['far']


        if far<far_min:

            if far<1/(86400*365):
                rate=1/far/(86400*365) #like 1 over 3 year
                s3='FAR=1/%s years' %round(rate,2)
            elif far<1/(86400*30) and far>=1/(86400*365):
                rate=1/far/(86400*30) #like 1 per x months
                s3='FAR=1/%s months' %round(rate,2)
            elif far>1/(86400*30):
                rate=1/far/(86400) #like 1 per x days
                s3='FAR=1/%s days' %round(rate,2)

            print(record['superevent_id'],'passes FAR cut:', s3)

            if record['event']['group'] == 'CBC':

                if record['event']['search'] == 'SSM':
                    print('SSM event detected, skipping p_astro classification ')
                    P_HasNS = round(record['event']['properties']['HasNS'],2)
                    P_HasMassGap = round(record['event']['properties']['HasMassGap'],2)
                    P_HasSSM = round(record['event']['properties']['HasSSM'],2)

                    print('P(HasNS):', P_HasNS, 'P(HasMassGap):', P_HasMassGap, 'P(HasSSM):', P_HasSSM)

                else:
                    BNS=round(record['event']['classification']['BNS'],2)
                    NSBH=round(record['event']['classification']['NSBH'],2)
                    BBH=round(record['event']['classification']['BBH'],2)
                    terr=round(record['event']['classification']['Terrestrial'],2)

                    print('Classifications - P(BNS):', BNS, 'P(NSBH):', NSBH, 'P(BBH):', BBH, 'P(Terr):', terr)

                    P_HasNS = round(record['event']['properties']['HasNS'],2)
                    P_HasRemnant = round(record['event']['properties']['HasRemnant'],2)
                    P_HasMassGap = round(record['event']['properties']['HasMassGap'],2)

                    print('P(HasNS):', P_HasNS, 'P(HasRemnant):', P_HasRemnant, 'P(HasMassGap):', P_HasMassGap)

            
            else:
                print('Selections passed, something else happened')

        else:
            print(record['superevent_id'],'does not pass FAR cut')

                
    except Exception as e:
        print("An error occurred while processing the record:", str(e))
        return


# Let's check past notices


In [4]:
import datetime
# Subscribe to topics and receive alerts
topics = 'igwn.gwalert'

timestamp1 = int(datetime.datetime(2025, 11, 12, 15, 31, 1, 501529, tzinfo=datetime.timezone.utc).timestamp() * 1000)
timestamp2 = int(timestamp1 + 0.5*86400000) # +2 days

start = consumer.offsets_for_times(
    [TopicPartition(topics, 0, timestamp1)])
end = consumer.offsets_for_times(
    [TopicPartition(topics, 0, timestamp2)])


consumer.assign(start)

# Calculate the number of messages to consume, ensuring it's within valid range
num_messages = end[0].offset - start[0].offset



for message in consumer.consume(abs(num_messages), timeout=2):
    
    if message.error():
        print(message.error())
        continue
    # Print the topic and message ID
    try:
        #print(f'topic={message.topic()}, offset={message.offset(), json.loads(message.value())["event"]["time"]}')
        parse_gw_notice(message.value())
        
    except Exception as e:
        print(f"Error parsing message: {e}")

    


Significant event detected: S251112cm
2025-11-13 16:57:17.783594 S251112cm SSM event
HasNS: 0.08 HasSSM: 1.0 HasMassGap: 0.0
S251112cm passes FAR cut: FAR=1/6.21 years
SSM event detected, skipping p_astro classification 
P(HasNS): 0.08 P(HasMassGap): 0.0 P(HasSSM): 1.0


%3|1763056678.473|FAIL|rdkafka#consumer-1| [thrd:sasl_ssl://kafka1.gcn.nasa.gov:9092/1]: sasl_ssl://kafka1.gcn.nasa.gov:9092/1: Receive failed: SSL transport error: Operation timed out (after 1108338ms in state UP)
%5|1763057357.075|REQTMOUT|rdkafka#consumer-1| [thrd:sasl_ssl://kafka3.gcn.nasa.gov:9092/3]: sasl_ssl://kafka3.gcn.nasa.gov:9092/3: Timed out MetadataRequest in flight (after 406275ms, timeout #0)
%4|1763057357.075|REQTMOUT|rdkafka#consumer-1| [thrd:sasl_ssl://kafka3.gcn.nasa.gov:9092/3]: sasl_ssl://kafka3.gcn.nasa.gov:9092/3: Timed out 1 in-flight, 0 retry-queued, 0 out-queue, 0 partially-sent requests
%5|1763057417.911|REQTMOUT|rdkafka#consumer-1| [thrd:sasl_ssl://kafka2.gcn.nasa.gov:9092/2]: sasl_ssl://kafka2.gcn.nasa.gov:9092/2: Timed out MetadataRequest in flight (after 60836ms, timeout #0)
%4|1763057417.911|REQTMOUT|rdkafka#consumer-1| [thrd:sasl_ssl://kafka2.gcn.nasa.gov:9092/2]: sasl_ssl://kafka2.gcn.nasa.gov:9092/2: Timed out 1 in-flight, 0 retry-queued, 0 out-queue